
# SyntaxError

Example script with invalid Python syntax



In [ ]:
"""
Metallicity prior choice moves the photometric posterior by ~0.3 dex
=====================================================================

Optical broadband photometry constrains metallicity weakly, so the
prior carries real information. We mock a star-forming galaxy at
log Z/Zsun = -0.5 in five SDSS bands at S/N=20, then fit it twice
under the same model — once with a uniform Z prior, once with a
Gaussian prior centred on 0 with sigma=0.3. The posteriors shift by
~0.3 dex toward each prior's preferred region, illustrating how
informative external priors propagate through a tengri inference.

Reference: Conroy 2013, ARA&A, 51, 393 (age-metallicity-dust
degeneracy in broadband fitting).
"""

import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

BANDS = ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]
ssp = tengri.load_ssp()
obs = tengri.Observation(photometry=tengri.Photometry.from_names(BANDS))

key = jax.random.PRNGKey(42)
<<<<<<< HEAD
truth = {
    "sfh_tsnorm_log_total_mass": 0.8,
=======
true_params = {
    "sfh_tsnorm_log_total_mass": 0.8,
>>>>>>> 22c20410 (refactor(sfh): complete repo-wide sweep of log_total_mass → log_total_mass)
    "sfh_tsnorm_peak_lbt_gyr": 2.0,
    "sfh_tsnorm_width_gyr": 1.5,
    "sfh_tsnorm_skew": 0.1,
    "sfh_tsnorm_trunc": 5.0,
    "met_logzsol": -0.5,
    "dust_tau_bc": 0.1,
    "dust_tau_diff": 0.2,
    "dust_slope": -0.7,
}
sfh_truth = {k.replace("sfh_tsnorm_", ""): v for k, v in truth.items() if k.startswith("sfh_")}
dust_truth = {k.replace("dust_", ""): v for k, v in truth.items() if k.startswith("dust_")}
template = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "met_logzsol": tengri.Fixed(truth["met_logzsol"]),
        **sfh_truth,
    },
    dust={"type": "two_component", "*": tengri.FIXED, **dust_truth},
    redshift=tengri.Fixed(0.1),
)
mock = template.mock(truth, snr=20.0, key=key)

<<<<<<< HEAD
=======
# Fit 2: Gaussian prior on metallicity (informative)
# NOTE: To use Gaussian priors, build with Parameters API then construct SEDModel
spec_gaussian = tengri.Parameters(
    sfh_tsnorm_log_total_mass=10.0, 2.5),
    sfh_tsnorm_peak_lbt_gyr=tengri.Uniform(0.5, 12.0),
    sfh_tsnorm_width_gyr=tengri.Uniform(0.3, 5.0),
    sfh_tsnorm_skew=tengri.Uniform(-3.0, 3.0),
    sfh_tsnorm_trunc=tengri.Uniform(1.0, 10.0),
    met_logzsol=tengri.Gaussian(mu=0.0, sigma=0.3),
    dust_tau_diff=tengri.Uniform(0.0, 1.5),
    dust_slope=tengri.Fixed(-0.7),
    redshift=tengri.Fixed(0.1),
    mean_sfh_type="tsnorm",
)
model_gaussian = tengri.SEDModel(spec_gaussian, ssp, observation=obs)
forward_g = tengri.ForwardModel.build(sed=model_gaussian, observation=obs)
post_g = forward_g.fit(
    mock.flux_obs, mock.noise, method="vi_native", n_iter=8, n_samples=3, verbose=False
)
>>>>>>> 22c20410 (refactor(sfh): complete repo-wide sweep of log_total_mass → log_total_mass)

def fit_with_metallicity_prior(met_prior):
    model = tengri.SEDModel.build(
        ssp,
        observation=obs,
        sfh={"type": "tsnorm", "*": tengri.FREE, "met_logzsol": met_prior},
        dust={
            "type": "two_component",
            "*": tengri.FIXED,
            "tau_diff": tengri.Uniform(0.0, 1.5),
            "slope": -0.7,
        },
        redshift=tengri.Fixed(0.1),
    )
    forward = tengri.ForwardModel.build(sed=model, observation=obs)
    return forward.fit(
        mock.flux_obs,
        mock.noise,
        method="native_vi_nonlinear",
        n_iterations=8,
        n_samples=3,
        verbose=False,
    )


post_u = fit_with_metallicity_prior(tengri.Uniform(-2.0, 0.2))
post_g = fit_with_metallicity_prior(tengri.Gaussian(mu=0.0, sigma=0.3))

met_u = np.asarray(post_u.samples["met_logzsol"])
met_g = np.asarray(post_g.samples["met_logzsol"])
z_grid = np.linspace(-2.0, 0.5, 200)
gauss_pdf = np.exp(-0.5 * (z_grid / 0.3) ** 2) / (0.3 * np.sqrt(2 * np.pi))

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6), sharey=True)
for ax, samples, prior_curve, prior_label, color in [
    (axes[0], met_u, np.ones_like(z_grid) / 2.2, "Uniform(-2, 0.2)", "C0"),
    (axes[1], met_g, gauss_pdf, r"Gaussian $\mathcal{N}(0, 0.3)$", "C3"),
]:
    ax.hist(samples, bins=20, density=True, color=color, alpha=0.5, label="Posterior")
    ax.plot(z_grid, prior_curve, "k--", lw=1.2, label="Prior")
    ax.axvline(truth["met_logzsol"], color="0.2", ls=":", lw=1.2, label="Truth")
    ax.set_xlabel(r"$\log_{10}(Z/Z_\odot)$")
    ax.text(0.04, 0.95, prior_label, transform=ax.transAxes, va="top", fontsize=9)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
axes[0].set_ylabel("Probability density")

fig.tight_layout()
plt.savefig("plot_recipe_compare_priors.png", dpi=150, bbox_inches="tight")